# Task 2: Setup, Evaluation Framework, and Baseline

This notebook prepares Task 2 only. It creates the fixed season split, fits shared normalisation statistics, defines the evaluation metrics, and evaluates the majority baseline.

## How to Run

Run this notebook before the three model notebooks. Its saved split is the only split used by Task 2.

## 1. Setup


In [ ]:
%matplotlib inline

import gc
import hashlib
import json
import math
import os
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b0, densenet121

from joblib import Parallel, delayed, dump as joblib_dump, load as joblib_load
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.*")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8b6fc0"]
MUTED = "#6b7280"


In [ ]:
# Find the repository root whether Jupyter starts in the root or this notebook folder.
import sys
from pathlib import Path

REPO_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")
sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import (
    MANIFEST, TEST_IMAGE_DIR, IMAGE_TARGET_SIZE, compute_normalisation,
    describe_split, load_image_array, load_manifest, make_split,
)
from src.task2_utils import (
    FEATURE_CONFIG, extract_visual_features,
    TASK2_OUTPUT_DIR, TASK2_PREDICTION_PATH, TASK2_SPLIT_PATH,
    calculate_run_fingerprint, ensure_task2_directories, export_validation_predictions,
)

ensure_task2_directories()
print("Repository root:", REPO_ROOT)
print("Manifest:", MANIFEST)
print("Image target size (w, h):", IMAGE_TARGET_SIZE)


### 1.1 Configuration

These settings control only the shared split and preprocessing. Model-specific training settings remain in each model notebook. Use the same `QUICK_RUN` value in Notebook 1 and every model notebook.


In [ ]:
TARGET = "season"
RANDOM_STATE = 42
QUICK_RUN = True
VALIDATION_SHARE = 0.20
FEATURE_N_JOBS = -1

if QUICK_RUN:
    print("QUICK_RUN: preparing a reduced training cache for workflow testing.")
else:
    print("FULL RUN: preparing all Task 2 training rows.")


#### Reuse notes

- `QUICK_RUN` prepares fewer training rows only for checking the workflow; its metrics are not reportable.
- Rerun this notebook after changing the dataset, split seed, validation share, image preprocessing or feature configuration.
- Otherwise, all three model notebooks reuse the saved arrays without repeating preprocessing.


## 2. Data and the Evaluation Framework

This section is fixed before model training. Every candidate is evaluated on the same validation rows with the same metrics.


### 2.1 Leakage-safe split

The shared `make_split` function keeps identical-image groups on one side of the split and stratifies by `season`. Classes with too few independent groups remain in training. The class mapping is fitted from training labels and saved with the final model.


In [ ]:
frame = load_manifest(TARGET)
print(f"Rows carrying an {TARGET} label: {len(frame):,}")
print(f"Distinct classes in the manifest: {frame[TARGET].nunique()}")

train_frame, val_frame = make_split(
    frame, TARGET, validation_share=VALIDATION_SHARE, random_state=RANDOM_STATE
)

split_train_frame = train_frame.copy()
split_val_frame = val_frame.copy()

if QUICK_RUN:
    # Stratified where possible; a plain sample is enough for a structural check.
    train_frame = train_frame.sample(n=min(5000, len(train_frame)), random_state=RANDOM_STATE)
    train_frame = train_frame.reset_index(drop=True)
    print("QUICK_RUN: training rows reduced to", len(train_frame))

display(describe_split(train_frame, val_frame, TARGET))

In [ ]:
# Label encoding. Fixed to the sorted training classes and exported for all model notebooks, because
# reconstructing it later from a different frame would silently permute every prediction.
CLASSES = sorted(train_frame[TARGET].unique())

if QUICK_RUN:
    # The subsample above can drop whole classes, which would leave validation rows with no
    # index to map to. Full runs never enter this branch: make_split keeps every class in training.
    keep = val_frame[TARGET].isin(CLASSES)
    print(f"QUICK_RUN: dropping {int((~keep).sum())} validation rows whose class was "
          "removed by the training subsample.")
    val_frame = val_frame.loc[keep].reset_index(drop=True)

CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
N_CLASSES = len(CLASSES)

# Any validation class absent from training cannot be predicted. make_split sends
# single-group classes to training, so this should be empty; the check is what proves it.
unseen = sorted(set(val_frame[TARGET]) - set(CLASSES))
assert not unseen, f"Validation holds classes never seen in training: {unseen}"

y_train = train_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()
y_val = val_frame[TARGET].map(CLASS_TO_INDEX).to_numpy()

train_support = pd.Series(np.bincount(y_train, minlength=N_CLASSES), index=CLASSES)
val_support = pd.Series(np.bincount(y_val, minlength=N_CLASSES), index=CLASSES)
SCOREABLE = np.flatnonzero(val_support.to_numpy() > 0)   # class indices macro averages use

print(f"Classes: {N_CLASSES}")
print(f"Scoreable in validation: {len(SCOREABLE)} | absent: {N_CLASSES - len(SCOREABLE)}")
print(f"Training support range: {train_support.max():,} down to {train_support.min()}")
print("Absent from validation:", sorted(np.array(CLASSES)[val_support.to_numpy() == 0]))

In [ ]:
class_table = pd.DataFrame({
    "Training images": train_support,
    "Validation images": val_support,
})
class_table["Training share %"] = class_table["Training images"] / len(y_train) * 100
display(class_table.style.format({"Training share %": "{:.1f}%"}))


### 2.2 Loading images into memory

The images are decoded once using the deterministic transform from notebook 00 and retained as `uint8`. Training augmentation is still sampled separately for every batch.


In [ ]:
DEVICE = torch.device("cpu")  # setup does not need the GPU


In [ ]:
def build_image_cache(frame, description):
    """Decode a frame's images once through the shared transform into one uint8 array.

    Only the deterministic transform from Section 3.1 of notebook 00 is applied here, exactly
    once per image. Augmentation still happens per epoch, so nothing about the training
    distribution is frozen by this cache.

    Returns:
        Array of shape (rows, height, width, 3), dtype uint8, in the frame's row order.
    """
    width, height = IMAGE_TARGET_SIZE
    images = np.empty((len(frame), height, width, 3), dtype=np.uint8)
    start = time.time()
    for position, path in enumerate(frame["path"]):
        images[position] = load_image_array(path, target_size=IMAGE_TARGET_SIZE, scale=False)
        if position and position % 10000 == 0:
            print(f"  {description}: {position:,} / {len(frame):,}")
    print(f"{description}: {len(frame):,} images in {time.time() - start:.0f}s "
          f"({images.nbytes / 1e6:.0f} MB)")
    return images


def report_memory(label=""):
    """Host RSS and, on CUDA, device allocation. Cheap, and it makes a leak visible early."""
    line = []
    try:
        import resource
        peak_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        line.append(f"host peak {peak_kb / 1e6:.2f} GB")
    except (ImportError, AttributeError):
        try:
            import psutil
            line.append(f"host RSS {psutil.Process().memory_info().rss / 1e9:.2f} GB")
        except ImportError:
            pass
    if DEVICE.type == "cuda":
        line.append(f"device allocated {torch.cuda.memory_allocated() / 1e9:.2f} GB "
                    f"reserved {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"[memory{' ' + label if label else ''}] " + " | ".join(line))


X_train_images = build_image_cache(train_frame, "train")
X_val_images = build_image_cache(val_frame, "validation")

assert len(X_train_images) == len(y_train) and len(X_val_images) == len(y_val)
report_memory("after caching")

### 2.3 Training-only normalisation

RGB mean and standard deviation are fitted on the Task 2 training rows only. The same constants are then applied to validation and test images.


In [ ]:
VERIFY_NORMALISATION = False   # True: re-decode from disk and assert the constants match

start = time.time()
# Sums are taken over the raw 0-255 values in float32 and divided by 255 at the end, which is
# the same statistic as scaling first: sum(x/255) is sum(x)/255, and the same for the squares.
# The reductions accumulate into float64, so the running totals stay exact at this scale while
# the working chunk stays float32. Promoting the chunk itself to float64 would cost four bytes
# per channel per pixel twice over, once for the chunk and once for its square.
total = np.zeros(3, dtype=np.float64)
total_square = np.zeros(3, dtype=np.float64)
n_pixels = 0
for begin in range(0, len(X_train_images), 2048):
    chunk = X_train_images[begin:begin + 2048].astype(np.float32)
    total += chunk.sum(axis=(0, 1, 2), dtype=np.float64)
    total_square += np.einsum("nhwc,nhwc->c", chunk, chunk, dtype=np.float64)
    n_pixels += chunk.shape[0] * chunk.shape[1] * chunk.shape[2]

mean_raw = total / n_pixels
variance_raw = np.maximum(total_square / n_pixels - mean_raw ** 2, 0.0)
NORM_MEAN = (mean_raw / 255.0).astype(np.float32)
NORM_STD = np.maximum(np.sqrt(variance_raw) / 255.0, 1e-6).astype(np.float32)

print(f"Fitted on {len(train_frame):,} training rows in {time.time() - start:.1f}s "
      "(from the cache, no second decode pass)")
print("Mean (R, G, B):", np.round(NORM_MEAN, 4))
print("Std  (R, G, B):", np.round(NORM_STD, 4))

if VERIFY_NORMALISATION:
    reference_mean, reference_std = compute_normalisation(train_frame,
                                                          target_size=IMAGE_TARGET_SIZE)
    print("Reference mean:", np.round(reference_mean, 4))
    print("Reference std: ", np.round(reference_std, 4))
    assert np.allclose(NORM_MEAN, reference_mean, atol=1e-4), "Mean disagrees with notebook 00"
    assert np.allclose(NORM_STD, reference_std, atol=1e-4), "Std disagrees with notebook 00"
    print("Verified against compute_normalisation.")

# White studio backgrounds dominate, so a mean near 0.9 is the expected result rather than a bug.
assert (NORM_MEAN > 0.5).all(), "Unexpectedly dark mean; check the transform before continuing."

### 2.4 Random Forest feature preprocessing

Random Forest cannot learn directly from image tensors. The shared visual feature extractor converts every prepared image into colour, texture, edge and foreground descriptors once. The saved matrices are reused by Notebook 2, so training does not repeat this work.


In [ ]:
def build_feature_cache(images, description):
    print(f"Extracting {description} visual features...")
    features = np.vstack(Parallel(n_jobs=FEATURE_N_JOBS)(
        delayed(extract_visual_features)(image) for image in images
    )).astype(np.float32)
    print(f"{description}: {features.shape[0]:,} rows x {features.shape[1]:,} features")
    return features

X_train_features = build_feature_cache(X_train_images, "training")
X_val_features = build_feature_cache(X_val_images, "validation")
assert len(X_train_features) == len(y_train) and len(X_val_features) == len(y_val)
print("Feature configuration:", FEATURE_CONFIG)


### 2.5 Metrics

Macro-F1 is the primary metric because every season should contribute equally. Accuracy, balanced accuracy and weighted F1 provide complementary views. Top-2 accuracy measures whether the correct season appears among two suggestions; top-5 is not meaningful for this small label space.


In [ ]:
def evaluate_predictions(y_true, y_pred, scores=None, name=""):
    labels = SCOREABLE
    row = {
        "Model": name,
        "Top-1 accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "Balanced accuracy": recall_score(y_true, y_pred, labels=labels,
                                             average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    if scores is not None:
        k = min(2, scores.shape[1])
        top_k = np.argpartition(scores, -k, axis=1)[:, -k:]
        row["Top-2 accuracy"] = np.mean([truth in choices for truth, choices in zip(y_true, top_k)])
    else:
        row["Top-2 accuracy"] = np.nan
    return pd.DataFrame([row])


def per_class_table(y_true, y_pred):
    rows = []
    for index in SCOREABLE:
        true_binary = y_true == index
        pred_binary = y_pred == index
        rows.append({
            "Season": CLASSES[index],
            "Support": int(true_binary.sum()),
            "Precision": (true_binary & pred_binary).sum() / max(pred_binary.sum(), 1),
            "Recall": (true_binary & pred_binary).sum() / max(true_binary.sum(), 1),
            "F1": f1_score(true_binary, pred_binary, zero_division=0),
        })
    return pd.DataFrame(rows)


RESULTS = []

def record(result_frame):
    RESULTS.append(result_frame)
    display(result_frame.style.format({c: "{:.4f}" for c in result_frame.columns if c != "Model"}))
    return result_frame


### 2.6 Simple baselines

The majority baseline measures what class frequency alone can achieve. The stratified-random baseline samples from the training prior and provides a second non-learning reference. Random Forest is treated as a candidate model in Section 3, not as a trivial baseline.


In [ ]:
majority_index = int(np.bincount(y_train, minlength=N_CLASSES).argmax())
majority_pred = np.full_like(y_val, majority_index)
majority_scores = np.zeros((len(y_val), N_CLASSES))
majority_scores[:, majority_index] = 1.0
print(f"Majority class: {CLASSES[majority_index]} "
      f"({train_support.iloc[majority_index]:,} training images)")
record(evaluate_predictions(y_val, majority_pred, majority_scores, "Baseline: majority class"))

prior = np.bincount(y_train, minlength=N_CLASSES) / len(y_train)
rng = np.random.RandomState(RANDOM_STATE)
stratified_pred = rng.choice(N_CLASSES, size=len(y_val), p=prior)
record(evaluate_predictions(
    y_val, stratified_pred, np.tile(prior, (len(y_val), 1)), "Baseline: stratified random"
))

## 3. Export Shared Task 2 Artefacts

In [ ]:
split_rows = pd.concat([
    pd.DataFrame({"id": train_frame["id"].astype(str), "split": "train",
                  "position": np.arange(len(train_frame))}),
    pd.DataFrame({"id": val_frame["id"].astype(str), "split": "validation",
                  "position": np.arange(len(val_frame))}),
], ignore_index=True)
TASK2_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
split_rows.to_csv(TASK2_SPLIT_PATH, index=False)

preprocessed_dir = REPO_ROOT / "preprocessed_datasets" / "task2"
preprocessed_dir.mkdir(parents=True, exist_ok=True)
def save_preprocessed_array(path, array, chunk_rows=1024):
    """Create an array export, or reuse an identical existing export.

    Model notebooks load these files as read-only memory maps. On Windows an open
    mapping cannot be truncated, so blindly calling np.save on a setup rerun raises
    OSError 22. Compare in bounded chunks and leave an identical file untouched.
    """
    path = Path(path)
    if path.exists():
        existing = np.load(path, mmap_mode="r")
        compatible = existing.shape == array.shape and existing.dtype == array.dtype
        identical = compatible and all(
            np.array_equal(existing[start:start + chunk_rows], array[start:start + chunk_rows])
            for start in range(0, len(array), chunk_rows)
        )
        del existing
        if identical:
            print("Reused unchanged preprocessed array:", path.name)
            return
        raise RuntimeError(
            f"Existing preprocessed array differs: {path}. Close Task 2 model kernels "
            "before replacing or regenerating prepared data."
        )
    with path.open("xb") as handle:
        np.save(handle, array)
    print("Saved preprocessed array:", path.name)

save_preprocessed_array(preprocessed_dir / "task2_deep_learning_train_images.npy", X_train_images)
save_preprocessed_array(preprocessed_dir / "task2_deep_learning_validation_images.npy", X_val_images)
save_preprocessed_array(preprocessed_dir / "task2_random_forest_train_features.npy", X_train_features)
save_preprocessed_array(preprocessed_dir / "task2_random_forest_validation_features.npy", X_val_features)

setup_dir = TASK2_OUTPUT_DIR / "setup"
setup_dir.mkdir(parents=True, exist_ok=True)
fingerprint = calculate_run_fingerprint(
    target=TARGET, classes=CLASSES, train_ids=train_frame["id"],
    validation_ids=val_frame["id"], random_state=RANDOM_STATE,
    image_target_size=IMAGE_TARGET_SIZE, normalisation_mean=NORM_MEAN,
    normalisation_std=NORM_STD, feature_version=FEATURE_CONFIG,
)
config = {
    "target": TARGET, "classes": CLASSES, "random_state": RANDOM_STATE, "quick_run": QUICK_RUN,
    "validation_share": VALIDATION_SHARE, "image_target_size": list(IMAGE_TARGET_SIZE),
    "normalisation_mean": NORM_MEAN.tolist(), "normalisation_std": NORM_STD.tolist(),
    "feature_config": FEATURE_CONFIG, "feature_count": int(X_train_features.shape[1]),
    "fingerprint": fingerprint,
    "train_ids": train_frame["id"].astype(str).tolist(),
    "validation_ids": val_frame["id"].astype(str).tolist(),
}
(setup_dir / "config.json").write_text(json.dumps(config, indent=2))
baseline_results = pd.concat(RESULTS, ignore_index=True)
baseline_results.to_json(setup_dir / "baseline_results.json", orient="records", indent=2)
export_validation_predictions(
    setup_dir / "baseline_validation_results.csv", val_frame, y_val,
    majority_pred, majority_scores, CLASSES,
)
with (setup_dir / "baseline_validation_scores.npz").open("wb") as handle:
    np.savez_compressed(
        handle, fingerprint=fingerprint, validation_ids=val_frame["id"].astype(str),
        true_indices=y_val, classes=np.asarray(CLASSES),
        majority_scores=majority_scores,
    )
print("Saved split:", TASK2_SPLIT_PATH)
print("Run fingerprint:", fingerprint)
print("Saved setup outputs:", setup_dir)

print("Saved preprocessed arrays:", preprocessed_dir)
